# Scratch CNN 

## Objective

This notebooks trains a Sctrach CNN model.
Training data is sourced from Data_generator notebooks that we have saved earlier

4 notebooks are used for training 
1 notebooks is used for validation 

Total number of samples used for training  are: `101771`

Total number of samples used for validation are: `25452`


# WandB setup

In [1]:
!pip install --upgrade wandb -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.3/25.3 MB 70.9 MB/s eta 0:00:00:00:0100:01


In [2]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f3003877 (23F3003877-t12026) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# Import Libraries

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
from IPython.display import Audio, display

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
import torchaudio
import torch.optim as optim 
import torchaudio.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

from glob import glob
import random
from tqdm import tqdm
import soundfile as sf

import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import os
import sys

from sklearn.metrics import f1_score , accuracy_score, classification_report, confusion_matrix
import math

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("github_access")
!git clone https://{secret_value_0}@github.com/Aryanch9797/dl-genai-project-26-t1.git

repo_path = "/kaggle/working/dl-genai-project-26-t1"
if repo_path not in sys.path:
    sys.path.append(repo_path)

from src.Trainers.custom_trainer import training_model
from src.dataset.test_dataset import test_mashed_dataset
from src.dataset.train_val_dataset import MelSpectrogramDataset
from src.models.scratch_CNN import CNN_mashup_model
from src.inference import prediction

device: cuda
Cloning into 'dl-genai-project-26-t1'...
remote: Enumerating objects: 226, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 226 (delta 47), reused 49 (delta 17), pack-reused 130 (from 1)
Receiving objects: 100% (226/226), 64.41 MiB | 42.14 MiB/s, done.
Resolving deltas: 100% (115/115), done.


# Data Loader

In [4]:
val_path ="/kaggle/input/data-generator-ipynb/train_data/"
train_paths = [
    "/kaggle/input/data-generator-2-ipynb/train_data/",
    "/kaggle/input/data-generator-3-ipynb/train_data/",
    "/kaggle/input/data-generator-4-ipynb/train_data/",
    "/kaggle/input/data-generator-5-ipynb/train_data/"
]
test_csv = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv" , index_col='id')
sample_submission = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/sample_submission.csv" , index_col='id' )
path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/"

In [5]:
train_dataset = MelSpectrogramDataset(False, train_paths, val_path)
val_dataset = MelSpectrogramDataset(True, train_paths, val_path)

train_loader = DataLoader(train_dataset, batch_size=124, shuffle=True, num_workers=4,pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=124, shuffle=False, num_workers=4,pin_memory=True)

test_data = test_mashed_dataset(path, test_csv, 16000, 10.24)
# batch_size 1 for test data so all audio chunks of one sample can be processed together
test_loader = DataLoader(test_data , batch_size=1, shuffle=False, num_workers=4,pin_memory=True)

In [6]:
label_to_genre = {
    0: "blues", 1: "classical", 2: "country", 3: "disco", 4: "hiphop",
    5: "jazz", 6: "metal", 7: "pop", 8: "reggae", 9: "rock"
}

In [7]:
wandb.init(
    project="Audio_Mashup",
    group="Scratch_CNN",  
    name="try_to_get_started_again", 
    config={
        "architecture": "Scratch_CNN_with_7_layers",
        "epochs": 40,
        "lr": 0.001,
        "dropout": 0.2,
        "audio_sr": 22050,
        "patience":5,
        "start_filters":32
    }
)

# CNN model

In [8]:
model = CNN_mashup_model(7,3,16000,0.2,32)
model.to(device)

CNN_mashup_model(
  (CNN_model): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Dropout(p=0.2, inplace=False)
    (5): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Dropout(p=0.2, inplace=False)
    (10): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU()
    (13): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (14): Dropout(p=0.2, inplace=False)
    (15): Conv2d(128, 256, kernel_size=(

# Model traning

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
loss_fn = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

model_trained = training_model(model, optimizer, train_loader, val_loader, 40, 5, loss_fn, scheduler,device )

  0%|          | 0/40 [00:00<?, ?it/s]

Epoch: 0, Training_f1_macro: 0.718614240719723, Training_accuracy: 0.720676813630602, Avg_train_loss: 0.7864019244945587


  2%|▎         | 1/40 [12:31<8:08:33, 751.64s/it]

Epoch: 0, validation_f1_macro: 0.5822940238286289, validation_accuracy: 0.5960631777463461


In [ ]:
import kagglehub

MODEL_UPLOAD_DIR = "/kaggle/working/Scratch_CNN" 
os.makedirs(MODEL_UPLOAD_DIR, exist_ok=True)

MODEL_SAVE_PATH = os.path.join(MODEL_UPLOAD_DIR, "Scratch_CNN.pth")
torch.save(model_trained.state_dict(), MODEL_SAVE_PATH)     # model_trained
print(f"Model saved to {MODEL_SAVE_PATH}")

KAGGLE_USERNAME = 'aryanchauhan97971234' 
MODEL = 'Scratch_CNN_3_4_V3'
FRAMEWORK = 'pytorch'
VARIATION = '7_3_22050_0.2_32'

handle = f'{KAGGLE_USERNAME}/{MODEL}/{FRAMEWORK}/{VARIATION}'

print(f"Uploading model from {MODEL_UPLOAD_DIR} to {handle}...")

kagglehub.model_upload(
    handle,                     
    MODEL_UPLOAD_DIR,           
    license_name="Apache 2.0", 
    version_notes="4th run with transfer, lr scheduler"
)
print("Upload complete!")